In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/mlfp05/ex_3.py ──
"""
# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 3: Shared Utilities
# ════════════════════════════════════════════════════════════════════════
#
# Common data loading, windowing, training, visualisation, and experiment
# tracking utilities shared across all technique files in this exercise.
#
# This module is NOT meant to be run standalone — import it from the
# technique files (01_vanilla_rnn.py, 02_lstm.py, etc.).
# ════════════════════════════════════════════════════════════════════════
"""

import asyncio
import pickle
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml.types import MetricSpec

# ── Constants ───────────────────────────────────────────────────────────
REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "stocks"
OUTPUT_DIR = REPO_ROOT / "outputs" / "mlfp05" / "ex_3"

TICKERS = {
    "^STI": "Straits Times Index",
    "D05.SI": "DBS Group Holdings",
    "9988.HK": "Alibaba HK",
    "AAPL": "Apple",
    "005930.KS": "Samsung",
    "7203.T": "Toyota",
}

SEQ_LEN = 20  # 20-day lookback (4 trading weeks)
FORECAST_HORIZON = 5  # predict next 5 days
FEATURES = ["Close", "High", "Low", "Volume"]
HIDDEN_DIM = 64
EPOCHS = 15
LR = 1e-3
CLIP = 1.0
BATCH_SIZE = 64


def init_environment() -> torch.device:
    """Set up environment, seeds, device, and output directories."""
    setup_environment()
    torch.manual_seed(42)
    np.random.seed(42)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    device = get_device()
    print(f"Using device: {device}")
    return device


# ── Data Loading ────────────────────────────────────────────────────────
def fetch_ticker(symbol: str) -> pl.DataFrame:
    """Download daily OHLCV bars from yfinance, return polars DataFrame."""
    import yfinance as yf

    df = yf.download(
        symbol, start="2010-01-01", end="2024-12-31", progress=False, auto_adjust=True
    )
    if df is None or len(df) == 0:
        raise RuntimeError(f"yfinance returned empty frame for {symbol}")
    df = df.copy()
    df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
    return pl.from_pandas(df.reset_index())


def load_or_fetch(symbol: str) -> tuple[pl.DataFrame | None, str]:
    """Load from parquet cache, or download and cache."""
    cache = DATA_DIR / f"{symbol.replace('^', '').replace('.', '_')}.parquet"
    if cache.exists():
        return pl.read_parquet(cache), "cache"
    try:
        df = fetch_ticker(symbol)
        df.write_parquet(cache)
        return df, "yfinance"
    except Exception as exc:
        print(f"  {symbol} unavailable ({type(exc).__name__}: {exc})")
        return None, "failed"


def load_stock_data() -> tuple[dict[str, pl.DataFrame], str, pl.DataFrame]:
    """Load all tickers and return (stock_data, primary_symbol, primary_df)."""
    stock_data: dict[str, pl.DataFrame] = {}
    for symbol, name in TICKERS.items():
        df, source = load_or_fetch(symbol)
        if df is not None:
            stock_data[symbol] = df
            print(f"  {symbol} ({name}): {len(df)} days [{source}]")

    if "^STI" not in stock_data and "AAPL" not in stock_data:
        raise RuntimeError("Need at least ^STI or AAPL data to proceed")

    primary = "^STI" if "^STI" in stock_data else "AAPL"
    primary_df = stock_data[primary]
    print(
        f"\nPrimary: {primary} -- {len(primary_df)} days, "
        f"{primary_df['Date'].min()} -> {primary_df['Date'].max()}"
    )
    return stock_data, primary, primary_df


# ── Windowed Datasets ───────────────────────────────────────────────────
def build_dataset(
    df: pl.DataFrame,
    seq_len: int = SEQ_LEN,
    horizon: int = FORECAST_HORIZON,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, int]:
    """Build (seq_len window) -> (next horizon closes) arrays with z-score normalisation.

    Returns: X, y, mean, std, n_train_windows
    """
    data = df.select(FEATURES).to_numpy().astype(np.float32)
    n = len(data)
    split_n = int(0.8 * n)
    train_data = data[:split_n]
    mean = train_data.mean(axis=0, keepdims=True)
    std = train_data.std(axis=0, keepdims=True) + 1e-8
    data_norm = (data - mean) / std

    n_windows = n - seq_len - horizon + 1
    X = np.stack([data_norm[i : i + seq_len] for i in range(n_windows)])
    y = np.stack(
        [data_norm[i + seq_len : i + seq_len + horizon, 0] for i in range(n_windows)]
    )
    split_idx = split_n - seq_len
    return X.astype(np.float32), y.astype(np.float32), mean, std, split_idx


def prepare_dataloaders(
    primary_df: pl.DataFrame,
    device: torch.device,
) -> tuple[
    DataLoader,
    DataLoader,
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    np.ndarray,
    np.ndarray,
    int,
    int,
]:
    """Build train/val dataloaders and return raw tensors plus normalisation stats.

    Returns: train_loader, val_loader, X_train_t, y_train_t, X_val_t, y_val_t,
             norm_mean, norm_std, n_train_w, n_features
    """
    X_all, y_all, norm_mean, norm_std, n_train_w = build_dataset(primary_df)
    print(
        f"Built {len(X_all)} windows (seq_len={SEQ_LEN}, horizon={FORECAST_HORIZON}); "
        f"train {n_train_w}, val {len(X_all) - n_train_w}"
    )

    X_train_t = torch.from_numpy(X_all[:n_train_w]).to(device)
    y_train_t = torch.from_numpy(y_all[:n_train_w]).to(device)
    X_val_t = torch.from_numpy(X_all[n_train_w:]).to(device)
    y_val_t = torch.from_numpy(y_all[n_train_w:]).to(device)
    print(f"  X_train: {tuple(X_train_t.shape)}  y_train: {tuple(y_train_t.shape)}")
    print(f"  X_val:   {tuple(X_val_t.shape)}    y_val:   {tuple(y_val_t.shape)}")

    train_loader = DataLoader(
        TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True
    )
    val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE)
    n_features = X_train_t.shape[-1]

    return (
        train_loader,
        val_loader,
        X_train_t,
        y_train_t,
        X_val_t,
        y_val_t,
        norm_mean,
        norm_std,
        n_train_w,
        n_features,
    )


# ── Experiment Tracking ─────────────────────────────────────────────────
async def _setup_engines(
    primary: str,
    experiment_suffix: str = "",
) -> tuple[ConnectionManager, ExperimentTracker, str, ModelRegistry | None, bool]:
    """Create ExperimentTracker (kailash-ml 1.1.1 factory) and ModelRegistry."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_rnns.db"
    registry_db = "sqlite:///mlfp05_rnns_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    exp_name = (
        f"m5_rnns_{primary}_{experiment_suffix}"
        if experiment_suffix
        else f"m5_rnns_{primary}"
    )
    return conn, tracker, exp_name, registry, True


def setup_engines(
    primary: str,
    experiment_suffix: str = "",
) -> tuple[ConnectionManager, ExperimentTracker, str, ModelRegistry | None, bool]:
    """Sync wrapper for engine setup."""
    return asyncio.run(_setup_engines(primary, experiment_suffix))


# ── Training Harness ────────────────────────────────────────────────────
def compute_gradient_norm(model: nn.Module) -> float:
    """Compute the total L2 norm of all gradients (before clipping)."""
    total_norm = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total_norm += p.grad.data.norm(2).item() ** 2
    return total_norm**0.5


def _predict(model: nn.Module, x: torch.Tensor, attn: bool = False) -> torch.Tensor:
    """Forward pass, handling attention models that return a tuple."""
    out = model(x)
    return out[0] if attn else out


async def _train_model_async(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    epochs: int = EPOCHS,
    lr: float = LR,
    clip: float = CLIP,
    attn: bool = False,
) -> dict[str, Any]:
    """Train with gradient tracking, log to ExperimentTracker."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    train_losses, val_losses, gradient_norms = [], [], []
    n_params = sum(p.numel() for p in model.parameters())

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(
            {
                "model_type": name,
                "hidden_dim": str(HIDDEN_DIM),
                "seq_len": str(SEQ_LEN),
                "forecast_horizon": str(FORECAST_HORIZON),
                "epochs": str(epochs),
                "lr": str(lr),
                "clip_norm": str(clip),
                "n_params": str(n_params),
            }
        )
        print(f"  [{name}] {n_params:,} parameters")

        for epoch in range(epochs):
            model.train()
            b_losses, e_grads = [], []
            for xb, yb in train_loader:
                opt.zero_grad()
                loss = F.mse_loss(_predict(model, xb, attn), yb)
                loss.backward()
                e_grads.append(compute_gradient_norm(model))
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip)
                opt.step()
                b_losses.append(loss.item())

            tl, gn = float(np.mean(b_losses)), float(np.mean(e_grads))
            train_losses.append(tl)
            gradient_norms.append(gn)

            model.eval()
            with torch.no_grad():
                vl = float(
                    np.mean(
                        [
                            F.mse_loss(_predict(model, xb, attn), yb).item()
                            for xb, yb in val_loader
                        ]
                    )
                )
            val_losses.append(vl)

            await run.log_metrics(
                {"train_loss": tl, "val_loss": vl, "gradient_norm": gn},
                step=epoch + 1,
            )
            print(
                f"  [{name}] epoch {epoch+1:2d}/{epochs}  "
                f"train={tl:.4f}  val={vl:.4f}  grad={gn:.4f}"
            )

        await run.log_metric("final_val_loss", val_losses[-1])

    return {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "gradient_norms": gradient_norms,
        "final_val_loss": val_losses[-1],
    }


def train_model(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    epochs: int = EPOCHS,
    lr: float = LR,
    clip: float = CLIP,
    attn: bool = False,
) -> dict[str, Any]:
    """Sync wrapper for training."""
    return asyncio.run(
        _train_model_async(
            model,
            name,
            tracker,
            exp_name,
            train_loader,
            val_loader,
            device,
            epochs,
            lr,
            clip,
            attn,
        )
    )


# ── Model Registry ──────────────────────────────────────────────────────
def register_best_model(
    model: nn.Module,
    model_name: str,
    val_loss: float,
    primary: str,
    registry: ModelRegistry | None,
    has_registry: bool,
) -> None:
    """Register a model in the ModelRegistry."""
    if not has_registry or registry is None:
        print("  ModelRegistry not available, skipping registration")
        return

    model_bytes = pickle.dumps(model.state_dict())
    # Architecture/ticker are encoded in the model name; numeric facts go to MetricSpec.
    reg_result = asyncio.run(
        registry.register_model(
            name=f"m5_rnn_{model_name.lower()}_{primary.replace('^', '')}",
            artifact=model_bytes,
            metrics=[
                MetricSpec(
                    name="val_loss", value=float(val_loss), higher_is_better=False
                ),
                MetricSpec(name="hidden_dim", value=float(HIDDEN_DIM)),
                MetricSpec(name="seq_len", value=float(SEQ_LEN)),
                MetricSpec(name="forecast_horizon", value=float(FORECAST_HORIZON)),
                MetricSpec(name="epochs", value=float(EPOCHS)),
            ],
        )
    )
    print(f"  Registered: {reg_result.name} v{reg_result.version}")


# ── Visualisation Helpers ───────────────────────────────────────────────
def get_visualizer() -> ModelVisualizer:
    """Create a ModelVisualizer instance."""
    return ModelVisualizer()


def plot_training_curves(
    viz: ModelVisualizer,
    results: dict[str, Any],
    model_name: str,
    output_prefix: str,
) -> None:
    """Plot training/validation loss curves and gradient norms."""
    train_metrics = {
        f"{model_name} train": results["train_losses"],
        f"{model_name} val": results["val_losses"],
    }
    viz.training_history(
        metrics=train_metrics, x_label="Epoch", y_label="MSE Loss"
    ).write_html(str(OUTPUT_DIR / f"{output_prefix}_training_curves.html"))

    grad_metrics = {model_name: results["gradient_norms"]}
    viz.training_history(
        metrics=grad_metrics, x_label="Epoch", y_label="Gradient L2 Norm"
    ).write_html(str(OUTPUT_DIR / f"{output_prefix}_gradient_norms.html"))


def plot_predictions(
    viz: ModelVisualizer,
    model: nn.Module,
    X_val_t: torch.Tensor,
    y_val_t: torch.Tensor,
    norm_mean: np.ndarray,
    norm_std: np.ndarray,
    output_prefix: str,
    attn: bool = False,
) -> tuple[np.ndarray, np.ndarray, torch.Tensor | None]:
    """Generate prediction-vs-actual scatter and return denormalised arrays.

    Returns: preds_denorm, actual_denorm, attn_weights (or None)
    """
    model.eval()
    with torch.no_grad():
        if attn:
            val_preds, attn_weights = model(X_val_t)
        else:
            val_preds = model(X_val_t)
            attn_weights = None

    close_mean, close_std = norm_mean[0, 0], norm_std[0, 0]
    preds_denorm = val_preds.cpu().numpy() * close_std + close_mean
    actual_denorm = y_val_t.cpu().numpy() * close_std + close_mean

    pred_df = pl.DataFrame(
        {
            "actual": actual_denorm[:, 0].tolist(),
            "predicted": preds_denorm[:, 0].tolist(),
        }
    )
    viz.scatter(pred_df, x="actual", y="predicted").write_html(
        str(OUTPUT_DIR / f"{output_prefix}_pred_vs_actual.html")
    )

    return preds_denorm, actual_denorm, attn_weights


def plot_time_series_overlay(
    preds_denorm: np.ndarray,
    actual_denorm: np.ndarray,
    output_prefix: str,
    title: str = "Predicted vs Actual Close Price",
    n_points: int = 200,
) -> None:
    """Plot predicted vs actual as overlaid time-series lines."""
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    n = min(n_points, len(preds_denorm))
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(
        range(n), actual_denorm[:n, 0], label="Actual", color="#2196F3", linewidth=1.5
    )
    ax.plot(
        range(n),
        preds_denorm[:n, 0],
        label="Predicted",
        color="#FF5722",
        linewidth=1.5,
        linestyle="--",
        alpha=0.85,
    )
    ax.set_xlabel("Validation Window Index")
    ax.set_ylabel("Close Price")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(str(OUTPUT_DIR / f"{output_prefix}_time_series_overlay.png"), dpi=150)
    plt.close(fig)
    print(f"  Saved: {output_prefix}_time_series_overlay.png")


def plot_horizon_error(
    preds_denorm: np.ndarray,
    actual_denorm: np.ndarray,
    model_name: str,
) -> list[float]:
    """Print and return RMSE by forecast horizon day."""
    print(f"\n  Forecast Error by Horizon Day ({model_name}):")
    rmses = []
    for day in range(FORECAST_HORIZON):
        rmse = (
            float(np.mean((preds_denorm[:, day] - actual_denorm[:, day]) ** 2)) ** 0.5
        )
        rmses.append(rmse)
        print(f"    Day {day + 1}: RMSE={rmse:.2f}")
    return rmses




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 3.1: Vanilla RNN for Sequence Prediction
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this section, you will be able to:
#   - Explain WHY sequences need memory (and why feedforward networks fail)
#   - Build a vanilla RNN with torch.nn.RNN for multi-step forecasting
#   - Measure gradient norms across timesteps to SEE vanishing gradients
#   - Track training with ExperimentTracker (per-epoch metrics)
#   - Visualise predicted vs actual time-series overlays
#
# PREREQUISITES: M5/ex_2 (CNNs, PyTorch training loops, batch norm).
# ESTIMATED TIME: ~25-30 min
#
# DATASET: STI + APAC/global stocks via yfinance (2010-2024).
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn



## THEORY — Why Sequences Need Memory


In [ ]:
#
# Imagine predicting tomorrow's stock price. A feedforward network sees
# today's numbers in isolation — it has no concept of "yesterday" or
# "last week." But markets have TRENDS: a stock rising for five straight
# days means something different from a stock that just jumped today.
#
# A Recurrent Neural Network (RNN) solves this by keeping a "hidden state"
# h_t that carries information from previous timesteps:
#
#   h_t = tanh(W_xh * x_t + W_hh * h_{t-1} + b)
#
# At each step, the RNN reads the new input x_t AND its own memory h_{t-1}.
# This is like reading a book: you understand each sentence because you
# remember what came before.
#
# THE CATCH — Vanishing Gradients:
# During backpropagation, gradients must flow backward through EVERY
# timestep. Each step multiplies the gradient by W_hh and squashes it
# through tanh. After 20-60 steps, the gradient either:
#   - Shrinks to near-zero (vanishing) — the network forgets early inputs
#   - Explodes to infinity (exploding) — training becomes unstable
#
# This is THE fundamental limitation of vanilla RNNs, and THE reason
# LSTMs and GRUs were invented (see 02_lstm.py and 03_gru.py).



In [ ]:
device = init_environment()



## TASK 1 — Load multi-stock data and build windowed datasets


In [ ]:
stock_data, PRIMARY, primary_df = load_stock_data()

(
    train_loader,
    val_loader,
    X_train_t,
    y_train_t,
    X_val_t,
    y_val_t,
    norm_mean,
    norm_std,
    n_train_w,
    N_FEATURES,
) = prepare_dataloaders(primary_df, device)



### Checkpoint 1


In [ ]:
assert len(stock_data) >= 2, f"Need >= 2 tickers, got {len(stock_data)}"
assert X_train_t.shape[1] == SEQ_LEN
assert y_train_t.shape[1] == FORECAST_HORIZON, "Multi-step target shape mismatch"
print("--- Checkpoint 1 passed --- data loaded and windowed\n")



## TASK 2 — Set up ExperimentTracker


In [ ]:
conn, tracker, exp_name, registry, has_registry = setup_engines(
    PRIMARY, experiment_suffix="vanilla_rnn"
)



### Checkpoint 2


In [ ]:
assert tracker is not None, "ExperimentTracker should be initialised"
assert exp_name is not None, "Experiment should be created"
print("--- Checkpoint 2 passed --- ExperimentTracker ready\n")



## TASK 3 — Build the Vanilla RNN


In [ ]:
# h_t = tanh(W_hh h_{t-1} + W_xh x_t + b)
# We use only the LAST hidden state h_T for prediction — all the
# information about the sequence must be compressed into one vector.
# This is a severe bottleneck for long sequences.
class VanillaRNN(nn.Module):
    def __init__(
        self, input_dim: int, hidden_dim: int, horizon: int = FORECAST_HORIZON
    ):
        super().__init__()
        # TODO: self.rnn — a single-layer tanh nn.RNN mapping input_dim features
        #   to hidden_dim units; inputs arrive as (batch, seq, features)
        # TODO: self.head — a linear layer from the hidden size to `horizon` outputs

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Run x through self.rnn — it returns (output sequence, final hidden);
        #   the output sequence has shape (batch, seq, hidden)
        # TODO: Feed only the LAST timestep's hidden vector to the head and
        #   return the forecast, shape (batch, horizon)
        pass


rnn_model = VanillaRNN(input_dim=N_FEATURES, hidden_dim=HIDDEN_DIM)
n_params = sum(p.numel() for p in rnn_model.parameters())
print(f"VanillaRNN: {n_params:,} parameters")



### Checkpoint 3


In [ ]:
dummy_input = torch.randn(2, SEQ_LEN, N_FEATURES, device=device)
rnn_model.to(device)
dummy_out = rnn_model(dummy_input)
assert dummy_out.shape == (
    2,
    FORECAST_HORIZON,
), f"Expected (2, {FORECAST_HORIZON}), got {dummy_out.shape}"
print("--- Checkpoint 3 passed --- VanillaRNN architecture verified\n")



## TASK 4 — Train the Vanilla RNN


In [ ]:
print(f"\n== Training VanillaRNN on {PRIMARY} ==")
rnn_results = train_model(
    rnn_model,
    "VanillaRNN",
    tracker,
    exp_name,
    train_loader,
    val_loader,
    device,
)



## DIAGNOSTIC CHECKPOINT — the vanishing-gradient textbook case


In [ ]:
# VanillaRNN is the classic vanishing-gradient architecture. Note what
# the pad can and cannot see: its gradient-flow reading is per
# PARAMETER TENSOR (weight_ih, weight_hh, head), aggregated over the
# whole sequence, so a recurrent layer can read HEALTHY even while the
# signal from early timesteps decays. The per-timestep decay is what
# the gradient_decay_rnn() measurement below shows directly.
from kailash_ml.diagnostics import run_diagnostic_checkpoint


def _mse_loss(m, batch):
    xb, yb = batch
    pred = m(xb)
    pred = pred[0] if isinstance(pred, tuple) else pred
    return nn.functional.mse_loss(pred, yb)


print("\n── Diagnostic Report (VanillaRNN) ──")
diag, findings = run_diagnostic_checkpoint(
    rnn_model,
    train_loader,
    _mse_loss,
    title="VanillaRNN",
    train_losses=rnn_results["train_losses"],
    val_losses=rnn_results["val_losses"],
    show=False,
)
print_prescription_pad(findings, "VanillaRNN")

# ══════ READING THE PRESCRIPTION PAD (key: see ex_1/01_standard_ae.py) ══════
# With train AND validation losses passed in, the loss-trend reading
# can flag over- or underfitting. If the pad is HEALTHY but the
# gradient-decay ratio below is large, you have learned the pad's
# blind spot: it measures layers, not time steps.



### Checkpoint 4


In [ ]:
assert len(rnn_results["train_losses"]) == EPOCHS, f"Should have {EPOCHS} epochs"
assert rnn_results["final_val_loss"] < 5.0, "Val loss suspiciously high"
print(f"\n  Final val loss: {rnn_results['final_val_loss']:.4f}")
print(f"  Avg gradient norm: {np.mean(rnn_results['gradient_norms']):.4f}")
print("--- Checkpoint 4 passed --- VanillaRNN trained\n")



## TASK 5 — Visualise: vanishing gradients in action


Extract gradient norms from a list of hidden states after backward().


Gradient norm at each timestep for a vanilla RNN.


In [ ]:
# We hand-roll an RNN step-by-step and measure the gradient norm at
# each timestep after backpropagating from the LAST step. If gradients
# vanish, the norm at step 0 will be orders of magnitude smaller than
# at step T — the network literally cannot learn from early inputs.


def _collect_grad_norms(hiddens: list[torch.Tensor]) -> list[float]:
    return [float(h.grad.norm().item()) if h.grad is not None else 0.0 for h in hiddens]


def gradient_decay_rnn(seq_len: int = 60) -> list[float]:
    torch.manual_seed(0)
    hd = 16
    # TODO: Create leaf tensors that track gradients: input-to-hidden weights
    #   W_xh (N_FEATURES, hd) and hidden-to-hidden W_hh (hd, hd), both random
    #   and scaled by 0.5, plus a zero bias b (hd)
    # TODO: Create a random input x of shape (1, seq_len, N_FEATURES) on device
    # TODO: Start from a zero hidden state h of shape (1, hd) that tracks gradients
    # TODO: Unroll the recurrence by hand for every timestep: the new h is
    #   tanh of (this step's input projected by W_xh + previous h projected by
    #   W_hh + b). Intermediate tensors drop their .grad by default — ask
    #   autograd to retain it, and collect every h in a list `hiddens`
    # TODO: Backpropagate a scalar built from the LAST hidden state only
    #   (sum of its squares), so earlier steps receive gradient through time
    # TODO: Return the per-step gradient norms via _collect_grad_norms
    pass


GRAD_SEQ_LEN = 60
rnn_decay = gradient_decay_rnn(seq_len=GRAD_SEQ_LEN)

rnn_ratio = rnn_decay[0] / max(rnn_decay[-1], 1e-12)
print(f"\n== Gradient Decay ({GRAD_SEQ_LEN} steps) ==")
print(
    f"  RNN:  first={rnn_decay[0]:.4e}  last={rnn_decay[-1]:.4e}  ratio={rnn_ratio:.4e}"
)
print("  Early timesteps receive near-zero gradient — the network forgets them.")

# TODO: Plot gradient decay on a log-scale y-axis (semilogy)
#   - x-axis: range(GRAD_SEQ_LEN) — timestep index
#   - y-axis: rnn_decay — gradient norm at each step
#   - Use ax.semilogy() for log scale
#   - Color: "#F44336", linewidth=2, label="Vanilla RNN"
#   - Add annotation at step 5 pointing to the low gradient region:
#     "Gradients vanish\nfor early timesteps"
#   - Save to OUTPUT_DIR / "01_rnn_gradient_decay.png"
fig, ax = plt.subplots(figsize=(12, 5))
# TODO: Plot and annotate the gradient decay curve
fig.tight_layout()
fig.savefig(str(OUTPUT_DIR / "01_rnn_gradient_decay.png"), dpi=150)
plt.close(fig)
print("  Saved: 01_rnn_gradient_decay.png")



### Checkpoint 5


In [ ]:
assert (
    rnn_decay[0] < rnn_decay[-1]
), "RNN should show vanishing gradients (early < late)"
print("--- Checkpoint 5 passed --- vanishing gradient problem demonstrated\n")



## TASK 6 — Visualise: predicted vs actual time-series overlay


Visualise how hidden states evolve across timesteps for a single sample.


In [ ]:
viz = get_visualizer()

# Training curves
plot_training_curves(viz, rnn_results, "VanillaRNN", "01_rnn")

# Prediction scatter + denormalised arrays
preds_denorm, actual_denorm, _ = plot_predictions(
    viz, rnn_model, X_val_t, y_val_t, norm_mean, norm_std, "01_rnn"
)

# Time-series overlay — the key visual proof of model behaviour
plot_time_series_overlay(
    preds_denorm,
    actual_denorm,
    "01_rnn",
    title=f"VanillaRNN: Predicted vs Actual Close ({PRIMARY})",
)


# Hidden state evolution — show how the RNN's memory changes over time
def plot_hidden_state_evolution(model: nn.Module, sample: torch.Tensor) -> None:
    model.eval()
    with torch.no_grad():
        # TODO: Get the rnn_layer from model.rnn
        # TODO: Initialise h = zeros(1, 1, HIDDEN_DIM) on device
        # TODO: Feed the sample to rnn_layer ONE timestep at a time (a length-1
        #   slice of the sequence), carrying h forward; after each step store
        #   h as a flat numpy vector in a list `hidden_states`
        pass

    # TODO: Stack hidden_states into a matrix of shape (seq_len, hidden_dim)
    hidden_matrix = None  # replace: one row per timestep

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

    # TODO: Plot heatmap of hidden_matrix.T using ax1.imshow()
    #   aspect="auto", cmap="RdBu_r", interpolation="nearest"
    #   Add colorbar, labels, title

    # TODO: Find top-5 most active dimensions by variance
    #   Plot each as a line on ax2 with labels

    fig.tight_layout()
    fig.savefig(str(OUTPUT_DIR / "01_rnn_hidden_state_evolution.png"), dpi=150)
    plt.close(fig)
    print("  Saved: 01_rnn_hidden_state_evolution.png")


sample_input = X_val_t[:1]  # single validation sample
plot_hidden_state_evolution(rnn_model, sample_input)

# Horizon error
rmses = plot_horizon_error(preds_denorm, actual_denorm, "VanillaRNN")



### Checkpoint 6


In [ ]:
assert (OUTPUT_DIR / "01_rnn_training_curves.html").exists()
assert (OUTPUT_DIR / "01_rnn_pred_vs_actual.html").exists()
assert (OUTPUT_DIR / "01_rnn_time_series_overlay.png").exists()
assert (OUTPUT_DIR / "01_rnn_hidden_state_evolution.png").exists()
print("--- Checkpoint 6 passed --- visualisations generated\n")



## TASK 7 — Register model


In [ ]:
register_best_model(
    rnn_model,
    "VanillaRNN",
    rnn_results["final_val_loss"],
    PRIMARY,
    registry,
    has_registry,
)



### Checkpoint 7


In [ ]:
assert rnn_results["final_val_loss"] < 5.0, "Val loss should be reasonable"
print("--- Checkpoint 7 passed --- model registered\n")



## APPLY — Singapore F&B Demand Forecasting


In [ ]:
#
# BUSINESS SCENARIO:
#   You are a data analyst at a Singapore kopitiam-style F&B chain
#   with 100+ outlets across the island.
#   Management wants to predict daily sales volume per outlet to
#   optimise ingredient ordering and staff scheduling.
#
# WHY A VANILLA RNN?
#   Daily F&B sales have SHORT-TERM patterns: weekday/weekend cycles,
#   lunch rushes, payday spikes. A 5-7 day lookback captures the weekly
#   cycle. Vanilla RNNs work well for short sequences — the vanishing
#   gradient problem only bites when you need to remember events from
#   30+ days ago.
#
# BUSINESS INTERPRETATION:
#   - Predicting 3 days ahead lets the chain order kaya, bread, and eggs
#     with the right lead time for its bakery and egg suppliers
#   - Overprediction: wasted ingredients (kaya toast bread has 2-day shelf life)
#   - Underprediction: stockouts during peak hours, lost revenue + customer churn
#   - At ~$8 average transaction across 100 outlets, even a 5% improvement
#     in demand forecasting accuracy saves ~$146K/year in waste reduction
print("=" * 70)
print("  APPLY: Singapore F&B Chain — Daily Demand Forecasting")
print("=" * 70)

# TODO: Generate realistic F&B sales data with weekly seasonality
#   - n_days = 365 * 3 (3 years)
#   - base_sales: weekly sinusoid (period 7 days) around 800 with amplitude 200
#   - Add weekend_boost (150 for days 5,6), payday_boost (100 for first 3 days of month)
#   - Add linear trend and Gaussian noise
#   - Clip minimum to 200
np.random.seed(42)
n_days = 365 * 3
# TODO: Build sales array with seasonality, trend, and noise

# TODO: Build windowed dataset with CAFE_SEQ=7, CAFE_HORIZON=3
#   - Normalise using training set mean/std
#   - Create windows of shape (n_windows, 7, 1) -> targets (n_windows, 3)
CAFE_SEQ = 7
CAFE_HORIZON = 3

# TODO: Create and train a VanillaRNN(input_dim=1, hidden_dim=32, horizon=CAFE_HORIZON)
#   - Use Adam optimizer with lr=1e-3
#   - Train for 20 epochs with gradient clipping at 1.0
#   - Evaluate on validation set

# TODO: Denormalise predictions and compute business metrics
#   - mae_day1: Mean Absolute Error for day-1 prediction
#   - accuracy_improvement: one minus the day-1 MAE relative to mean actual sales (floor 0)
#   - waste reduction and annual savings calculation (store as annual_savings)

# TODO: Plot cafe demand forecast vs actual (60-day window)
#   - Actual as solid blue line, predicted as dashed orange
#   - Save to OUTPUT_DIR / "01_rnn_fnb_demand_forecast.png"



### Checkpoint 8 (Apply)


In [ ]:
assert mae_day1 < 200, "Day-1 MAE should be reasonable for F&B demand"
assert (OUTPUT_DIR / "01_rnn_fnb_demand_forecast.png").exists()
print("--- Checkpoint 8 passed --- Singapore F&B application complete\n")



## REFLECTION


[x] Built a vanilla RNN for multi-step stock forecasting ({SEQ_LEN}-day -> {FORECAST_HORIZON}-day)
  [x] Demonstrated vanishing gradients: gradient ratio = {rnn_ratio:.4e}
  [x] Hidden state evolution: how the RNN's memory changes across timesteps
  [x] Predicted vs actual time-series overlay (visual proof of model behaviour)
  [x] Tracked training with ExperimentTracker (loss + gradient norms)
  [x] Applied RNN to Singapore F&B demand forecasting (synthetic sales)
  [x] Quantified business impact: S${annual_savings:,.0f}/year savings

  Key insight: Vanilla RNNs work for SHORT sequences (5-10 steps) where
  recent patterns dominate. For longer dependencies — stock trends over
  months, seasonal patterns, patient histories — gradients vanish and
  the model forgets. That is exactly what LSTMs solve.

  Next: 02_lstm.py — how gates solve the vanishing gradient problem.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

